[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/10-agentic-ai/notebooks/03-langgraph-agents.ipynb)

# Notebook 03 — LangGraph: state, nodes, edges, memory and human approval

**Goal:** learn LangGraph's building blocks by running small graphs, then build a tool-calling agent with memory and a human-approval step. Checked against **langgraph 1.2.x**.

Vocabulary (also in concepts.md):

* **State** — a dictionary passed through the graph.
* **Node** — a function: takes state, returns an *update* (only the keys it changes).
* **Edge** — a fixed arrow. **Conditional edge** — an arrow chosen at run time.
* **Reducer** — how an update merges into state (replace, or e.g. append).
* **Checkpointer** — saves state after every step; **thread_id** names one saved conversation.

All offline: the "LLM" node is a stand-in function.

## Step 0 — Check the installed version

In Colab you may need `pip install langgraph` first. The cell does that only if the import fails.

In [1]:
try:
    import langgraph
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langgraph"], check=True)
from importlib.metadata import version
print("langgraph", version("langgraph"), "| langchain-core", version("langchain-core"))

langgraph 1.2.24 | langchain-core 1.8.4


## Step 1 — The smallest graph: two nodes in a line

State has two fields. Each node returns only the field it changes. `START` and `END` are the entry and exit.

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    text: str
    word_count: int

def clean(state: State) -> dict:
    return {"text": state["text"].strip().lower()}

def count(state: State) -> dict:
    return {"word_count": len(state["text"].split())}

g = StateGraph(State)
g.add_node("clean", clean)
g.add_node("count", count)
g.add_edge(START, "clean")
g.add_edge("clean", "count")
g.add_edge("count", END)
app = g.compile()
print(app.invoke({"text": "   Agents Are LLMs In A Loop  ", "word_count": 0}))

{'text': 'agents are llms in a loop', 'word_count': 6}


## Step 2 — Reducers: append instead of replace

By default a node's update **replaces** the old value. With `Annotated[list, operator.add]` the update is **added** (appended) to the list instead. This is how chat history grows.

In [3]:
import operator
from typing import Annotated

class LogState(TypedDict):
    log: Annotated[list, operator.add]     # reducer = list addition (append)

def a(state): return {"log": ["a ran"]}
def b(state): return {"log": ["b ran"]}

g = StateGraph(LogState)
g.add_node("a", a); g.add_node("b", b)
g.add_edge(START, "a"); g.add_edge("a", "b"); g.add_edge("b", END)
print(g.compile().invoke({"log": ["start"]}))

{'log': ['start', 'a ran', 'b ran']}


## Step 3 — Conditional edges and loops

A **router function** looks at the state and returns the name of the next node. Here a "draft" node keeps improving a draft until a quality score is high enough — a loop, with a safety limit.

In [4]:
class DraftState(TypedDict):
    draft: str
    score: int
    rounds: int

def write(state):
    return {"draft": state["draft"] + " +detail", "score": state["score"] + 30, "rounds": state["rounds"] + 1}

def route(state) -> str:
    if state["score"] >= 80 or state["rounds"] >= 5:   # good enough, or safety stop
        return "publish"
    return "write"

def publish(state):
    return {"draft": state["draft"] + " [PUBLISHED]"}

g = StateGraph(DraftState)
g.add_node("write", write); g.add_node("publish", publish)
g.add_edge(START, "write")
g.add_conditional_edges("write", route, {"write": "write", "publish": "publish"})
g.add_edge("publish", END)
loop_app = g.compile()
print(loop_app.invoke({"draft": "Intro", "score": 0, "rounds": 0}))

{'draft': 'Intro +detail +detail +detail [PUBLISHED]', 'score': 90, 'rounds': 3}


`stream` shows each node's update as it happens — this is how UIs show progress.

In [5]:
for update in loop_app.stream({"draft": "Intro", "score": 0, "rounds": 0}, stream_mode="updates"):
    print(update)

{'write': {'draft': 'Intro +detail', 'score': 30, 'rounds': 1}}
{'write': {'draft': 'Intro +detail +detail', 'score': 60, 'rounds': 2}}
{'write': {'draft': 'Intro +detail +detail +detail', 'score': 90, 'rounds': 3}}
{'publish': {'draft': 'Intro +detail +detail +detail [PUBLISHED]'}}


LangGraph can describe its own graph as Mermaid text (the same diagram language used in this course's images).

In [6]:
print(loop_app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	write(write)
	publish(publish)
	__end__([<p>__end__</p>]):::last
	__start__ --> write;
	write -.-> publish;
	publish --> __end__;
	write -.-> write;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 4 — A tool-calling agent with prebuilt pieces

* `MessagesState` = state with a `messages` list that **appends** automatically.
* `ToolNode(tools)` runs every tool call found in the last AI message.
* `tools_condition` routes to `"tools"` if there are tool calls, otherwise to `END`.

The `llm` node is an offline stand-in that emits a tool call for new questions and answers once it sees a tool result.

In [7]:
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return {"taipei": "Sunny, 24C", "london": "Rainy, 9C"}.get(city.lower(), "Cloudy, 20C")

TOOLS = [get_weather]

def fake_llm(state: MessagesState):
    last = state["messages"][-1]
    if isinstance(last, HumanMessage):
        q = last.content.lower()
        if "weather" in q:
            city = "Taipei" if "taipei" in q else "London"
            return {"messages": [AIMessage(content="", tool_calls=[{"name": "get_weather", "args": {"city": city}, "id": f"call_{len(state['messages'])}"}])]}
        # questions about earlier conversation: look back through history (memory!)
        earlier = [m.content for m in state["messages"] if isinstance(m, ToolMessage)]
        return {"messages": [AIMessage(content=f"Earlier tool results I remember: {earlier}")]}
    return {"messages": [AIMessage(content=f"The weather is: {last.content}")]}

g = StateGraph(MessagesState)
g.add_node("llm", fake_llm)
g.add_node("tools", ToolNode(TOOLS))
g.add_edge(START, "llm")
g.add_conditional_edges("llm", tools_condition)
g.add_edge("tools", "llm")
print("graph built:", list(g.nodes))

graph built: ['llm', 'tools']


## Step 5 — Memory with a checkpointer

Compile with `InMemorySaver()`. Each call with the same `thread_id` continues the same conversation, because the saved state (all messages) is loaded first. A different `thread_id` starts fresh.

In [8]:
from langgraph.checkpoint.memory import InMemorySaver

agent = g.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "student-1"}}

r1 = agent.invoke({"messages": [HumanMessage("What's the weather in Taipei?")]}, cfg)
for m in r1["messages"]:
    print(f"{type(m).__name__:12}", repr(m.content), getattr(m, "tool_calls", "") or "")

HumanMessage "What's the weather in Taipei?" 
AIMessage    '' [{'name': 'get_weather', 'args': {'city': 'Taipei'}, 'id': 'call_1', 'type': 'tool_call'}]
ToolMessage  'Sunny, 24C' 
AIMessage    'The weather is: Sunny, 24C' 


Second turn, **same thread**: the agent can see the first turn.

In [9]:
r2 = agent.invoke({"messages": [HumanMessage("What did you find out before?")]}, cfg)
print(r2["messages"][-1].content)
print("messages saved in this thread:", len(r2["messages"]))

Earlier tool results I remember: ['Sunny, 24C']
messages saved in this thread: 6


A **different thread** has no memory of the first one.

In [10]:
r3 = agent.invoke({"messages": [HumanMessage("What did you find out before?")]}, {"configurable": {"thread_id": "student-2"}})
print(r3["messages"][-1].content)

Earlier tool results I remember: []


You can inspect the saved state and its history (one checkpoint per step) — very useful for debugging.

In [11]:
snap = agent.get_state(cfg)
print("next node to run:", snap.next)
print("number of checkpoints:", len(list(agent.get_state_history(cfg))))

next node to run: ()
number of checkpoints: 8


## Step 6 — Human-in-the-loop with `interrupt`

Calling `interrupt(payload)` inside a node **pauses** the graph and saves it. The caller sees the payload under `__interrupt__`. Later we resume with `Command(resume=answer)`; the node continues and `interrupt` returns that answer. A checkpointer is required (the paused state must be stored).

In [12]:
from langgraph.types import interrupt, Command

class RefundState(TypedDict):
    order_id: str
    amount: int
    status: str

def propose(state):
    return {"status": f"proposed refund of {state['amount']} for {state['order_id']}"}

def human_gate(state):
    if state["amount"] <= 100:                       # small refunds: no approval needed
        return {"status": state["status"] + " | auto-approved"}
    answer = interrupt({"question": f"Approve {state['status']}?"})
    return {"status": state["status"] + (" | APPROVED" if answer == "yes" else " | REJECTED")}

def execute(state):
    done = "APPROVED" in state["status"] or "auto-approved" in state["status"]
    return {"status": state["status"] + (" | refund sent" if done else " | nothing sent")}

g = StateGraph(RefundState)
for name, fn in [("propose", propose), ("human_gate", human_gate), ("execute", execute)]:
    g.add_node(name, fn)
g.add_edge(START, "propose"); g.add_edge("propose", "human_gate"); g.add_edge("human_gate", "execute"); g.add_edge("execute", END)
refund_app = g.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "refund-42"}}
paused = refund_app.invoke({"order_id": "A42", "amount": 900, "status": ""}, cfg)
print("Paused with:", paused["__interrupt__"][0].value)
print("Waiting at node:", refund_app.get_state(cfg).next)

Paused with: {'question': 'Approve proposed refund of 900 for A42?'}
Waiting at node: ('human_gate',)


A human (in a real app: a button in a web page) answers **yes**. We resume the same thread.

In [13]:
final = refund_app.invoke(Command(resume="yes"), cfg)
print(final["status"])

small = refund_app.invoke({"order_id": "B7", "amount": 40, "status": ""}, {"configurable": {"thread_id": "refund-7"}})
print(small["status"])

proposed refund of 900 for A42 | APPROVED | refund sent
proposed refund of 40 for B7 | auto-approved | refund sent


## Step 7 — Optional: a real model

With `ANTHROPIC_API_KEY` set and `pip install langchain-anthropic`, the stand-in can be replaced by a real chat model with tools bound. The graph does not change.

In [14]:
import os
if os.getenv("ANTHROPIC_API_KEY"):
    try:
        from langchain_anthropic import ChatAnthropic
        model = ChatAnthropic(model=os.getenv("ANTHROPIC_MODEL", "claude-opus-5-5")).bind_tools(TOOLS)
        g2 = StateGraph(MessagesState)
        g2.add_node("llm", lambda s: {"messages": [model.invoke(s["messages"])]})
        g2.add_node("tools", ToolNode(TOOLS))
        g2.add_edge(START, "llm"); g2.add_conditional_edges("llm", tools_condition); g2.add_edge("tools", "llm")
        real = g2.compile(checkpointer=InMemorySaver())
        out = real.invoke({"messages": [HumanMessage("Weather in London?")]}, {"configurable": {"thread_id": "real"}})
        print(out["messages"][-1].content)
    except ImportError:
        print("pip install langchain-anthropic to try the real model")
else:
    print("No ANTHROPIC_API_KEY - real model skipped.")

No ANTHROPIC_API_KEY - real model skipped.


## Recap

| Piece | Code |
|---|---|
| State | `class State(TypedDict)` / `MessagesState` |
| Node | `g.add_node("name", fn)` |
| Edge | `g.add_edge("a", "b")` |
| Conditional edge | `g.add_conditional_edges("a", router, {...})` |
| Memory | `compile(checkpointer=InMemorySaver())` + `thread_id` |
| Human approval | `interrupt(...)` then `invoke(Command(resume=...))` |

**Try it:** add a `max_refund` check node that routes amounts over 5000 straight to a "reject" node without asking a human.